# DuckDB-backed catalogs at scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jejjohnson/geotoolz/blob/main/docs/notebooks/catalog_duckdb.ipynb)

`DuckDBGeoCatalog` is the Phase-2 backend: a lazy SQL relation over a
GeoParquet artifact. Same `GeoCatalog` Protocol as
`InMemoryGeoCatalog`, but the rows live on disk (or in S3 / GCS /
HuggingFace) and queries push down to the Parquet reader so you read
only the row groups your AOI touches.

This notebook builds a small catalog, persists it as GeoParquet,
reopens it through DuckDB, and walks the Protocol surface:
`query`, `intersect`, `union`, `iter_rows`, `materialize`.

In [1]:
import subprocess
import sys


try:
    import google.colab  # noqa: F401

    on_colab = True
except ImportError:
    on_colab = False

if on_colab:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "geotoolz[duckdb]"])

In [2]:
import geopandas as gpd
import pandas as pd
import shapely.geometry

import geotoolz as gz

## A small in-memory catalog

We start from a hand-rolled `InMemoryGeoCatalog` of two tiles in
UTM zone 29N — small enough to fit in RAM, but the same surface
scales to 10⁶ rows once we route through DuckDB.

In [3]:
gdf = gpd.GeoDataFrame(
    {
        "geometry": [
            shapely.geometry.box(0, 0, 100, 100),
            shapely.geometry.box(200, 0, 300, 100),
        ],
        "start_time": [pd.Timestamp("2024-01-01"), pd.Timestamp("2024-01-02")],
        "end_time": [pd.Timestamp("2024-01-02"), pd.Timestamp("2024-01-03")],
        "filepath": ["A.tif", "B.tif"],
    },
    geometry="geometry",
    crs="EPSG:32629",
)
mem = gz.InMemoryGeoCatalog(gdf, backend="raster")
mem

InMemoryGeoCatalog(backend='raster', len=2, crs=<Projected CRS: EPSG:32629>
Name: WGS 84 / UTM zone 29N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 12°W and 6°W, northern hemisphere between equator and 84°N, onshore and offshore. Algeria. Côte D'Ivoire (Ivory Coast). Faroe Islands. Guinea. Ireland. Jan Mayen. Liberia, Mali. Mauritania. Morocco. Portugal. Sierra Leone. Spain. United Kingdom (UK). Western Sahara.
- bounds: (-12.01, 0.0, -6.0, 84.01)
Coordinate Operation:
- name: UTM zone 29N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich
)

## Persist as GeoParquet, reopen via DuckDB

`to_geoparquet` writes the catalog with the GeoParquet 1.1 bbox
covering struct, which DuckDB uses for predicate pushdown.
`open_catalog(path)` is the factory — it prefers the DuckDB backend
when `[duckdb]` is installed and silently falls back to the in-memory
backend otherwise.

In [4]:
import pathlib
import tempfile


tmp = pathlib.Path(tempfile.mkdtemp())
gz.to_geoparquet(mem, tmp / "cat.parquet")

duck = gz.open_catalog(tmp / "cat.parquet")
duck

DuckDBGeoCatalog(backend='raster', crs='EPSG:32629')

`len`, `total_bounds`, `temporal_extent` work just like the
in-memory backend — but each is one SQL aggregate, not a Python
loop.

In [5]:
print("rows           :", len(duck))
print("total_bounds   :", duck.total_bounds)
print("temporal_extent:", duck.temporal_extent)
print("config         :", duck.get_config())

rows           : 2
total_bounds   : (0.0, 0.0, 300.0, 100.0)
temporal_extent: [2024-01-01 00:00:00, 2024-01-03 00:00:00]
config         : {'backend': 'raster', 'len': 2, 'crs': 'EPSG:32629', 'engine': 'duckdb'}


## Spatial + temporal queries push down

A `GeoSlice` carries bounds + interval + CRS together. Passing one to
`query` translates to a SQL `WHERE` clause that DuckDB can push down
to the Parquet reader.

In [6]:
sl = gz.GeoSlice(
    bounds=(0, 0, 50, 50),
    interval=pd.Interval(
        pd.Timestamp("2024-01-01"),
        pd.Timestamp("2024-01-02"),
        closed="both",
    ),
    resolution=(1.0, 1.0),
    crs="EPSG:32629",
)
hits = duck.query(sl)
print("matched", len(hits), "row(s):")
hits.materialize().gdf

matched 1 row(s):


,filepath,_backend,_schema_version,bbox,geometry
datetime,,,,,
"[2024-01-01 00:00:00, 2024-01-02 00:00:00]",A.tif,raster,0,"{'xmin': 0.0, 'ymin': 0.0, 'xmax': 100.0, 'yma...","POLYGON ((100 0, 100 100, 0 100, 0 0, 100 0))"


### Cross-CRS queries reproject internally

An AOI in EPSG:4326 against a UTM-zone-29N catalog used to silently
return zero rows in earlier homebrew catalogs (the §10.1 footgun in
the design plan). The DuckDB backend reprojects the AOI before the
SQL is built, so the right rows come back.

In [7]:
# UTM 29N (50, 50) ≈ (-13.488°, 0.00045°) in 4326.
duck.query(
    bounds=(-13.4885, 0.0001, -13.4880, 0.0008), crs="EPSG:4326"
).materialize().gdf

,filepath,_backend,_schema_version,bbox,geometry
datetime,,,,,
"[2024-01-01 00:00:00, 2024-01-02 00:00:00]",A.tif,raster,0,"{'xmin': 0.0, 'ymin': 0.0, 'xmax': 100.0, 'yma...","POLYGON ((100 0, 100 100, 0 100, 0 0, 100 0))"


## Set algebra: intersect + union as SQL joins

`intersect` is a SQL spatial join clipped to `ST_Intersection`;
`union` is `UNION ALL`. Both return new lazy relations.

In [8]:
labels = gz.InMemoryGeoCatalog(
    gpd.GeoDataFrame(
        {
            "geometry": [shapely.geometry.box(50, 50, 250, 150)],
            "start_time": [pd.Timestamp("2024-01-01")],
            "end_time": [pd.Timestamp("2024-01-04")],
            "filepath": ["labels.gpkg"],
        },
        geometry="geometry",
        crs="EPSG:32629",
    ),
    backend="vector",
)

joint = duck.intersect(labels)
joint.materialize().gdf

,filepath,geometry
datetime,,
"[2024-01-01 00:00:00, 2024-01-02 00:00:00]",A.tif,"POLYGON ((100 100, 100 50, 50 50, 50 100, 100 ..."
"[2024-01-02 00:00:00, 2024-01-03 00:00:00]",B.tif,"POLYGON ((200 100, 250 100, 250 50, 200 50, 20..."


In [9]:
merged = duck.union(labels)
print("rows after union:", len(merged))

rows after union: 3


## `iter_rows` — the streaming surface

Loaders and the `geotoolz.patch` bridge consume `CatalogRow`
instances. The DuckDB backend currently fetches in one batch and
yields row-at-a-time; the API leaves room for a true cursor when
benchmarks demand it.

In [10]:
for row in duck.iter_rows():
    print(row.filepath, "—", row.geometry.bounds, "—", row.interval)

A.tif — (0.0, 0.0, 100.0, 100.0) — [2024-01-01 00:00:00, 2024-01-02 00:00:00]
B.tif — (200.0, 0.0, 300.0, 100.0) — [2024-01-02 00:00:00, 2024-01-03 00:00:00]


## `materialize` — back to a GeoDataFrame when you need one

When the rest of your pipeline expects a `GeoDataFrame`, pull the
relation eagerly. Useful at the boundary between the catalog layer
and a pandas-flavoured analytics step.

In [11]:
mat = duck.materialize()
mat.gdf

,filepath,_backend,_schema_version,bbox,geometry
datetime,,,,,
"[2024-01-01 00:00:00, 2024-01-02 00:00:00]",A.tif,raster,0,"{'xmin': 0.0, 'ymin': 0.0, 'xmax': 100.0, 'yma...","POLYGON ((100 0, 100 100, 0 100, 0 0, 100 0))"
"[2024-01-02 00:00:00, 2024-01-03 00:00:00]",B.tif,raster,0,"{'xmin': 200.0, 'ymin': 0.0, 'xmax': 300.0, 'y...","POLYGON ((300 0, 300 100, 200 100, 200 0, 300 0))"


## When to use DuckDB

- Catalog scale past ~10⁵ rows (RAM ceiling for the gdf backend).
- The catalog needs to be portable — a colleague queries it without
  rebuilding.
- You want cloud-hosted catalogs (`s3://bucket/cat.parquet`);
  DuckDB's `httpfs` reads only the row groups your query touches.
- You want full SQL escape-hatch power (`.sql("…")`).

Stick with `InMemoryGeoCatalog` for prototyping, small training-set
construction, or when you don't want the `[duckdb]` dependency.